# Exemples d'usage de la base de données

In [ ]:
import os
#os.environ['USE_PYGEOS'] = '0'
from fradiodb import *
import pandas as pd
import geopandas as gpd
import folium
import matplotlib as mpl
DB='anfr.db'

In [ ]:
#download_data()
#import_anfr_zip(DB)

## Exemple 1 : récupérer quelques données

In [ ]:
cnx = sqlite3.connect(DB)
display(pd.read_sql_query("select * from sites limit 3", cnx))
display(pd.read_sql_query("select * from supports limit 3", cnx))
display(pd.read_sql_query("select * from antennas limit 3", cnx))
display(pd.read_sql_query("select * from stations limit 3", cnx))
display(pd.read_sql_query("select * from transmitters limit 3", cnx))

## Exemple 2 : Réseaux mobiles 900 MHz et GSM-R
Pour filtrer les sites par techno, on va utiliser le champ `tech_bitmask` : pour cela, ouvrir `bitmask.ods` et insérer `1` dans le champ ON/OFF sur les technologies souhaitées : les champs `bitmask1` et `bitmask2` sont ensuite utilisables dans une requête SQL : typiquement pour faire un OU logique sur les technos sélectionnées, on fera `... and (tech_bitmask1 & XXX) > 0` afin de retenir l'ensemble des sites sur lesquels au moins 1 bit est à 1 i.e. qui contient au moins une des technos sélectionnées dans le masque. De même, il est possible d'exclure des technologies avec `... and (tech_bitmask1 & YYY) = 0`. Les champs `tech_bitmask1` et `tech_bitmask2` sont disponibles dans les tables `antennas/transmitters/supports/sites` (et les vues associées). Il faut évidemment choisir la table ou la vue en fonction du point de vue qu'on souhaite avoir (e.g. `sites` pour avoir juste les emplacements sous forme de point, `antennas` pour disposer de l'azimut, etc. En gardant en tête qu'un site inclue plusieurs supports, incluant plusieurs antennes, incluant plusieurs transmetteurs...)

Imaginons qu'on souhaite récupérer l'ensemble des sites d'opérateurs télécoms 900 MHz et GSM-R un peu comme dans le [rapport ECC 318](https://docdb.cept.org/download/1433) en affichant différemment les sites d'opérateurs et ceux de la SNCF
* dans le cas des réseaux mobiles, on va choisir {"GSM 900", "LTE 900", "UMTS 900"}, ce qui donne bitmask1=1101659111424 et bitmask2=1
* une seule techno gsm-r pour le moment (en attendant le FRMCS) ce qui donne bitmask1=4294967296 (et bitmask2=0)

Dans l'exemple ci-dessous, on voit clairement le tracé du GSM-R suivant les lignes de chemin de fer.

In [ ]:
gsmr_sites = pd.read_sql_query("select * from sites where (tech_bitmask1 & 4294967296)>0", cnx)
gdf_gsmr = gpd.GeoDataFrame(gsmr_sites, geometry=gpd.points_from_xy(x=gsmr_sites.lon, y=gsmr_sites.lat), crs=4326)

rop_sites = pd.read_sql_query("select * from sites where ((tech_bitmask1 & 1101659111424)>0 or (tech_bitmask2 & 1)>0) and (lat>42 and lat<52 and lon>-5 and lon<9)", cnx)
gdf_rop = gpd.GeoDataFrame(rop_sites, geometry=gpd.points_from_xy(x=rop_sites.lon, y=rop_sites.lat), crs=4326)

m=gdf_rop.explore(style_kwds=dict(color="black", weight=1, opacity=0.4, s=1))
gdf_gsmr.explore(m=m, style_kwds=dict(color="orange", weight=2, opacity=1, s=2))

## Example 2 : Sites 5G
Affichons ci-dessous l'ensemble des sites 5G 3500 MHz ("vraie 5G") par opérateur : pour cela c'est simple, le bitmask correspondant est 32. Attention : en écrivant `tech_bitmask1=32` on ne garderait que les sites incluant strictement et uniquement cette techno. Pour faire un "OU logique" i.e. garder les sites incluant de la 5G 3.5 GHz ainsi que d'autres équipements, il faut écrire `(tech_bitmask1 & 32)>0`. Si on souhaite exclure une technologie (par exemple exclure les sites 5G 700 MHz), on peut ajouter une condition séparée e.g. `and (tech_bitmask1 & 128)=0`.

Garder en tête que
* un site peut inclure plusieurs supports avec plusieurs stations, chacune avec plusieurs émetteurs, chacun relié à plusieurs antennes
* mais du fait du RAN sharing (MORAN/MOCN), cela peut aussi se compliquer e.g. une antenne peut être reliée à plusieurs émetteurs ou à plusieurs stations

In [ ]:
def sortops(operator_list_str):
    operator_list_str = operator_list_str.replace(" TELECOM", "").replace(" MOBILE", "")
    ops = operator_list_str.split(',')
    if len(ops)==1: return operator_list_str
    if len(ops)==2 and "SFR" in ops and "BOUYGUES" in ops: return "BOUYGES+SFR"
    return f"_MIXED {len(ops)} ops" #','.join(sorted(ops))
cnx.create_function("sortops", 1, sortops)

query = """select lon, lat,
    count(distinct operator) operator_count, sortops(group_concat(distinct operator)) operator_list,
    max(ant_height) h_max,
    cast(group_concat(distinct station_id) as text) stations,
    cast(group_concat(distinct system) as text) tech_list,
    sites.tech_bitmask1, sites.tech_bitmask2
    from sites
    inner join antennas on antennas.sup_id=supports.id
    inner join supports on sites.id=supports.site_id
    inner join transmitters on transmitters.antenna_id=antennas.id
    inner join id_systems on transmitters.system_id=id_systems.id
    inner join stations on stations.id=transmitters.station_id
    inner join id_operators on id_operators.id=stations.operator_id

    where (antennas.tech_bitmask1 & 32)>0 and (lat>42 and lat<52 and lon>-5 and lon<10)
    group by sites.id;
"""
rop_sites = pd.read_sql_query(query, cnx)
gdf_5G = gpd.GeoDataFrame(rop_sites, geometry=gpd.points_from_xy(x=rop_sites.lon, y=rop_sites.lat), crs=4326)
cmap_ops = mpl.colors.ListedColormap(['magenta', 'blue', 'green', 'orange', 'red', 'silver', 'gray', 'black'])
gdf_5G.explore(column='operator_list', cmap=cmap_ops) #opacity=0.5, fill_opacity=0.5, s=2, weight=1, 

Affichons maintenant les sites tous opérateurs confondus, avec une colormap par techno plutôt que par opérateur. N.B. j'ai classé les technos subjectivement (les dénominations "4G+", "5G+" ne correspondent pas à la définition des opérateurs)
* j'ai par exemple mis la 5G 700 MHz dans "4G" car les débits sont analogues à la 4G sur cette même bande (canal de 2x10 MHz, pas d'AAS...)
* il y a peut-être des AAS 5G à 1800 MHz et 2100 MHz, néanmoins j'ai estimé que seule la bande 3.5 GHz permettait débits réellement différenciant

On voit sur la carte ci-dessous les zones denses où de fortes capacités 4G+/5G ont été déployées. On voit aussi que dans les zones rurales, la quasi-totalité des sites supporte désormais la 4G (dans les bandes basses)

In [ ]:
imt_tech_order = {'GSM 900': '2G', 'GSM 1800': '2G',
                 'UMTS 900': '3G', 'UMTS 2100': '3G',
                 'LTE 900': '4G', 'LTE 700': '4G', 'LTE 800': '4G', '5G NR 700': '4G',
                 'LTE 1800': '4G', 'LTE 2100': '4G', 'LTE 2600': '4G+',
                 '5G NR 1800': '4G+', '5G NR 2100': '4G+', '5G NR 3500': '5G'}
def best_tech(tech_list_str):
    tech_list_index = [imt_tech_order[k] for k in tech_list_str.split(',') if k in imt_tech_order]
    return max(tech_list_index)
cnx.create_function("best_tech", 1, best_tech)

query = """select lon, lat,
    max(ant_height) h_max,
    group_concat(distinct system) tech_list,
    best_tech(group_concat(distinct system)) tech,
    sites.tech_bitmask1, sites.tech_bitmask2
    from sites
    inner join antennas on antennas.sup_id=supports.id
    inner join supports on sites.id=supports.site_id
    inner join transmitters on transmitters.antenna_id=antennas.id
    inner join id_systems on transmitters.system_id=id_systems.id
    inner join stations on stations.id=transmitters.station_id

    where ( (antennas.tech_bitmask1 & 2047625658540)>0 or (antennas.tech_bitmask2 & 3072)>0)
    and (lat>42 and lat<52 and lon>-5 and lon<10)
    group by sites.id;
"""
rop_sites = pd.read_sql_query(query, cnx)
gdf_5G = gpd.GeoDataFrame(rop_sites, geometry=gpd.points_from_xy(x=rop_sites.lon, y=rop_sites.lat), crs=4326)
gdf_5G.explore(column='tech', cmap='RdYlGn') #RdYlGn brg

## Example 3 : Radiodiffusion
Exemple ci-dessous en choisissant les sites de radiodiffusion (radio et TV) : on voit clairement les nombreux petits sites FM le long des autoroutes, et la densité de sites plus importante en zone montagneuse

In [ ]:
def sortechs(tech_list_str):
    techs = tech_list_str.split(',')
    if len(techs)==1: return tech_list_str
    return ','.join(sorted(techs))
cnx.create_function("sortechs", 1, sortechs)

query2 = """select lon, lat,
    group_concat(distinct supports.id) supports, min(ant_height) h_min,
    sortechs(group_concat(distinct system)) tech_list,
    sites.tech_bitmask1, sites.tech_bitmask2
    from sites
    inner join supports on sites.id=supports.site_id
    inner join antennas on antennas.sup_id=supports.id
    inner join transmitters on transmitters.antenna_id=antennas.id
    inner join id_systems on transmitters.system_id=id_systems.id

    where (lat>42 and lat<52 and lon>-5 and lon<9)
    and (antennas.tech_bitmask1 & 131941428887552)>0
    group by sites.id;
"""
fm_sites = pd.read_sql_query(query2, cnx)
gdf_fm = gpd.GeoDataFrame(fm_sites, geometry=gpd.points_from_xy(x=fm_sites.lon, y=fm_sites.lat), crs=4326)
gdf_fm.explore(column='tech_list')